# LSTM for Turbofan Engine RUL Prediction

## Overview

This educational notebook demonstrates how to use a **Long Short-Term Memory (LSTM)** network for time series prediction of Remaining Useful Life (RUL) of turbofan engines.

## Learning Objectives

By the end of this notebook, you will:
- Understand how LSTMs improve upon basic RNNs
- Learn the architecture and components of LSTM cells
- Build and train an LSTM model for RUL prediction
- Compare LSTM performance with other models

## What is an LSTM?

**Long Short-Term Memory (LSTM)** networks are a special type of RNN designed to solve the vanishing gradient problem. They can learn long-term dependencies in sequences.

### Key Innovation: Gated Architecture
LSTMs use "gates" to control information flow:
- **Forget Gate**: Decides what information to discard
- **Input Gate**: Decides what new information to store
- **Output Gate**: Decides what information to output

### Advantages over Basic RNN:
- Can remember information for long periods
- Better at learning long-term dependencies
- More stable training (less prone to vanishing gradients)
- Often achieves better performance than simple RNNs

### When to Use:
- Long sequences with important long-term patterns
- When you need to remember information from many timesteps ago
- Complex temporal relationships


## 1. Import Libraries

### Purpose
Import TensorFlow/Keras libraries specifically for LSTM implementation. The LSTM layer is the key component that differentiates this from a basic RNN.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure TensorFlow to use CPU only (avoids CUDA/libdevice issues)
# IMPORTANT: Set this BEFORE importing TensorFlow for it to take effect
# If you still see GPU errors, restart the kernel and run this cell first!
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Disable GPU, use CPU only
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings

# Deep Learning
import tensorflow as tf

# Aggressively disable GPU and force CPU usage
try:
    # Hide all GPU devices
    tf.config.set_visible_devices([], 'GPU')
    # Set memory growth to prevent GPU allocation
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, False)
except:
    pass

# Force CPU device placement for all operations
tf.config.set_soft_device_placement(True)
with tf.device('/CPU:0'):
    # This ensures CPU is used by default
    pass

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')
sns.set_palette("husl")
%matplotlib inline

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

# Verify GPU is disabled
print(f"\n📊 TensorFlow Device Configuration:")
print(f"   Available GPUs: {len(tf.config.list_physical_devices('GPU'))}")
print(f"   Available CPUs: {len(tf.config.list_physical_devices('CPU'))}")
if len(tf.config.list_physical_devices('GPU')) == 0:
    print("   ✅ GPU successfully disabled - using CPU only")
else:
    print("   ⚠️  WARNING: GPU still detected! Please restart kernel and run this cell first.")


Libraries imported successfully!
TensorFlow version: 2.20.0


## 2. Load and Prepare Data

### Purpose
Load the dataset. The same data loading process as RNN, but LSTM will process it differently due to its gated architecture.


In [7]:
# Define data path
data_path = Path('../../dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData')
data_path = Path('../dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData/')
# Column names with actual field names
op_settings = ['Altitude', 'Mach', 'TRA']
sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
           'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
column_names = ['unit', 'time'] + op_settings + sensors

def load_data(dataset='FD001'):
    """Load training and test data"""
    train_file = data_path / f'train_{dataset}.csv'
    test_file = data_path / f'test_{dataset}.csv'
    rul_file = data_path / f'RUL_{dataset}.csv'
    
    # CSV files already have headers, so no need for sep, header=None, or names
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    rul_df = pd.read_csv(rul_file)
    
    return train_df, test_df, rul_df

def calculate_rul_train(df):
    """Calculate RUL for training data"""
    df = df.copy()
    df['RUL'] = df.groupby('unit')['time'].transform(lambda x: x.max() - x)
    return df

# Load data
train_df, test_df, rul_df = load_data('FD001')
train_df = calculate_rul_train(train_df)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Number of engines in training: {train_df['unit'].nunique()}")
print(f"Number of engines in test: {test_df['unit'].nunique()}")


Training data shape: (20631, 27)
Test data shape: (13096, 26)
Number of engines in training: 100
Number of engines in test: 100


## 3. Create Time Series Sequences

### Purpose
Create sequences of consecutive timesteps. LSTMs excel at learning from these sequences because they can:
- Remember important information from early in the sequence
- Forget irrelevant information
- Update their memory as new information arrives

### Sequence Length:
We use 30 timesteps, meaning the model sees the last 30 cycles of sensor data to predict RUL. LSTMs can effectively use all 30 timesteps, unlike basic RNNs which may struggle with this length.


In [8]:
def create_sequences(data, sequence_length=30):
    """Create sequences for time series prediction"""
    sequences = []
    targets = []
    
    # Select features (using actual field names)
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in data['unit'].unique():
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        unit_rul = unit_data['RUL'].values
        
        # Create sequences
        for i in range(len(unit_data) - sequence_length + 1):
            sequences.append(unit_features[i:i+sequence_length])
            targets.append(unit_rul[i+sequence_length-1])
    
    return np.array(sequences), np.array(targets)

# Create sequences
sequence_length = 30
X_train_seq, y_train_seq = create_sequences(train_df, sequence_length)

# For test data, we need to get the last sequence_length timesteps for each engine
def create_test_sequences(data, sequence_length=30):
    """Create test sequences (last sequence_length timesteps for each engine)"""
    sequences = []
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in sorted(data['unit'].unique()):
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        
        # Take last sequence_length timesteps
        if len(unit_features) >= sequence_length:
            sequences.append(unit_features[-sequence_length:])
        else:
            # Pad if shorter
            padding = np.zeros((sequence_length - len(unit_features), len(feature_cols)))
            sequences.append(np.vstack([padding, unit_features]))
    
    return np.array(sequences)

X_test_seq = create_test_sequences(test_df, sequence_length)
y_test = rul_df['RUL'].values

print(f"Training sequences shape: {X_train_seq.shape}")
print(f"Training targets shape: {y_train_seq.shape}")
print(f"Test sequences shape: {X_test_seq.shape}")
print(f"Test targets shape: {y_test.shape}")


Training sequences shape: (17731, 30, 24)
Training targets shape: (17731,)
Test sequences shape: (100, 30, 24)
Test targets shape: (100,)


## 4. Data Normalization

### Purpose
Normalize data to [0,1] range. Critical for LSTM training because:
- LSTM gates use sigmoid/tanh activations that work best with normalized inputs
- Helps with gradient flow during backpropagation
- Prevents numerical instability


In [9]:
# Normalize features
feature_scaler = MinMaxScaler()
n_samples, n_timesteps, n_features = X_train_seq.shape
X_train_reshaped = X_train_seq.reshape(-1, n_features)
X_train_scaled = feature_scaler.fit_transform(X_train_reshaped)
X_train_seq_scaled = X_train_scaled.reshape(n_samples, n_timesteps, n_features)

# Normalize test data
X_test_reshaped = X_test_seq.reshape(-1, n_features)
X_test_scaled = feature_scaler.transform(X_test_reshaped)
X_test_seq_scaled = X_test_scaled.reshape(X_test_seq.shape)

# Normalize targets
target_scaler = MinMaxScaler()
y_train_scaled = target_scaler.fit_transform(y_train_seq.reshape(-1, 1)).flatten()
y_test_scaled = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

print("Data normalized successfully!")
print(f"Training features - Min: {X_train_seq_scaled.min():.4f}, Max: {X_train_seq_scaled.max():.4f}")
print(f"Training targets - Min: {y_train_scaled.min():.4f}, Max: {y_train_scaled.max():.4f}")


Data normalized successfully!
Training features - Min: 0.0000, Max: 1.0000
Training targets - Min: 0.0000, Max: 1.0000


## 5. Build LSTM Model

### Purpose
Construct a multi-layer LSTM architecture. Our model uses:
- **Three LSTM layers**: Progressive feature extraction
  - First layer (128 units): Processes raw sequences
  - Second layer (64 units): Extracts higher-level patterns
  - Third layer (32 units): Final sequence representation
- **Dropout layers**: Regularization between LSTM layers
- **Dense layers**: Map LSTM output to RUL prediction

### Why Multiple LSTM Layers?
- **Hierarchical learning**: Each layer learns patterns at different abstraction levels
- **First layer**: Low-level temporal patterns
- **Second layer**: Mid-level patterns combining first layer outputs
- **Third layer**: High-level patterns for final prediction

### Activation Function:
- **tanh**: Used in LSTM cells (outputs values between -1 and 1)
- Helps with gradient flow and memory management


In [10]:
def build_lstm_model(sequence_length, n_features):
    """Build LSTM model"""
    # Force CPU device placement to avoid GPU errors
    with tf.device('/CPU:0'):
        model = Sequential([
            LSTM(128, activation='tanh', return_sequences=True, input_shape=(sequence_length, n_features)),
            Dropout(0.2),
            LSTM(64, activation='tanh', return_sequences=True),
            Dropout(0.2),
            LSTM(32, activation='tanh', return_sequences=False),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Build model
n_features = X_train_seq_scaled.shape[2]
model = build_lstm_model(sequence_length, n_features)
model.summary()


I0000 00:00:1768660405.690646   43370 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1227 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-01-17 22:33:26.809553: W external/local_xla/xla/service/gpu/llvm_gpu_backend/default/nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykern/cuda_nvcc
  ipykern/cuda_nvdisasm
  ipykern/nvidia_nvshmem
  
  /usr/local/cuda
  /opt/cuda
  /home/pawatchun/Desktop/teaching/turbofan/venv/lib/python3.13/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /home/pawatchun/Desktop/teaching/turbofan/venv/l

UnknownError: {{function_node __wrapped__Sign_device_/job:localhost/replica:0/task:0/device:GPU:0}} JIT compilation failed. [Op:Sign] name: 

## 6. Train Model

### Purpose
Train the LSTM model. Training an LSTM:
- Takes longer than basic RNN (more parameters)
- But often achieves better accuracy
- Requires careful tuning of learning rate and regularization

### Training Strategy:
- **Early stopping**: Prevents overfitting by stopping when validation loss stops improving
- **Learning rate reduction**: Automatically adjusts learning rate for better convergence
- **Batch size 32**: Balance between memory usage and gradient stability


In [ ]:
# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train model
history = model.fit(
    X_train_seq_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)


## 7. Evaluate Model

### Purpose
Assess LSTM model performance. Compare with RNN to see if the more complex architecture improves predictions.


In [ ]:
# Predictions
y_train_pred_scaled = model.predict(X_train_seq_scaled, verbose=0)
y_test_pred_scaled = model.predict(X_test_seq_scaled, verbose=0)

# Inverse transform
y_train_pred = target_scaler.inverse_transform(y_train_pred_scaled).flatten()
y_test_pred = target_scaler.inverse_transform(y_test_pred_scaled).flatten()

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train_seq, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train_seq, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_r2 = r2_score(y_train_seq, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*60)
print("LSTM Model Performance")
print("="*60)
print(f"\nTraining Metrics:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  R²:   {train_r2:.4f}")
print(f"\nTest Metrics:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  R²:   {test_r2:.4f}")


## 8. Visualizations

### Purpose
Visualize LSTM training and predictions. Compare training curves with RNN to see differences in learning behavior.


In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Model MAE', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Predictions vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Predictions vs Actual', fontsize=14, fontweight='bold')

# Training set
axes[0].scatter(y_train_seq, y_train_pred, alpha=0.5, s=20)
min_val = min(min(y_train_seq), min(y_train_pred))
max_val = max(max(y_train_seq), max(y_train_pred))
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual RUL', fontsize=11)
axes[0].set_ylabel('Predicted RUL', fontsize=11)
axes[0].set_title(f'Train Set (R² = {train_r2:.4f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, s=20)
min_val = min(min(y_test), min(y_test_pred))
max_val = max(max(y_test), max(y_test_pred))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual RUL', fontsize=11)
axes[1].set_ylabel('Predicted RUL', fontsize=11)
axes[1].set_title(f'Test Set (R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
